In [1]:
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import copy
import random
import csv
import rings_utils

import torch
import torch.nn as nn
import torch.distributions as dist
from torch.utils.data import random_split
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
# 定義路徑
base_dir = 'rings_generation'
file_name = 'rings_algo_test.csv'
file_path = os.path.join(base_dir, file_name)
print(f"File Path: {file_path}")

# 確保路徑存在
if not os.path.exists(file_path):
    raise FileNotFoundError(f"The file {file_path} does not exist.")

# 載入數據
data = []
with open(file_path, 'r') as file:
    reader = csv.reader(file)
    for row in reader:
        data.append([float(value) for value in row])

data = np.array(data)
print(f"Data Shape: {data.shape}")

# 轉換為 PyTorch Tensor
dataset = torch.tensor(data, dtype=torch.float).to(device)

# 切分數據集
total_size = len(dataset)
train_size = int(0.8 * total_size)
test_size = total_size - train_size

# 使用 random_split 分割數據集
dataset_train, dataset_test = random_split(dataset, [train_size, test_size])

# 印出結果
print(f"Train Dataset Size: {len(dataset_train)}")
print(f"Test Dataset Size: {len(dataset_test)}")

File Path: rings_generation/rings_algo_test.csv
Data Shape: (10000, 512)
Train Dataset Size: 8000
Test Dataset Size: 2000


# Model

In [4]:
class CategoricalVAE(nn.Module):
    
    def __init__(self, input_layer, hidden_layers, N, K):
        
        super().__init__()

        self.N = N
        self.K = K
        
        # Encoder
        layer = []
        in_features = input_layer
        for out_features in hidden_layers:
            layer.append( nn.Linear(in_features, out_features) )
            layer.append( nn.ReLU() )
            in_features = out_features
        layer.append( nn.Linear(in_features, self.N*self.K) )
        self.encoder = nn.Sequential(*layer)
        
        # Decoder
        layer = []
        in_features = self.N*self.K
        for out_features in reversed(hidden_layers):
            layer.append( nn.Linear(in_features, out_features) )
            layer.append( nn.ReLU() )
            in_features = out_features
        layer.append( nn.Linear(in_features, input_layer) )
        layer.append( nn.Sigmoid() )
        self.decoder = nn.Sequential(*layer)
        
    def encode(self, inputs):
        '''
        input
        - inputs.shape: [batch_size, struct_size]
        output
        - logits.shape: [batch_size, N*K]
        '''
        logits = self.encoder(inputs)
        return logits

    def decode(self, latent_codes):
        '''
        input
        - latent_codes.shape: [batch_size, N*K]
        output
        - outputs.shape: [batch_size, struct_size]
        '''
        outputs = self.decoder(latent_codes)
        return outputs

    def gumbel_dist(self, shape, eps=1e-20):
        '''
        input
        - shape.shape: [batch_size, N*K]
        output
        - gumbel_noise.shape: [batch_size, N*K]
        '''
        U = torch.rand(shape).to(device)
        gumbel_noise = -torch.log(-torch.log(U + eps) + eps)
        return gumbel_noise

    def gumbel_softmax(self, logits, temperature):
        '''
        input
        - logits.shape: [batch_size, N*K]
        - temperature.shape: [1]
        output
        - latent_codes.shape: [batch_size, N*K]
        '''
        # Reshape to [batch_size*N, K]
        batch_size = len(logits)
        logits = logits.view(batch_size * self.N, self.K)
        gumbel_noise = self.gumbel_dist(logits.shape)
        y = logits + gumbel_noise
        latent_codes = nn.functional.softmax(y / temperature, dim=-1)

        # Reshape back to [batch_size, N*K]
        logits = logits.view(batch_size, self.N * self.K)
        latent_codes = latent_codes.view(batch_size, self.N * self.K)

        return latent_codes
    
    def forward(self, inputs, temperature):
        '''
        input
        - inputs.shape: [batch_size, struct_size]
        output
        - logits.shape: [batch_size, N*K]
        - latent_codes.shape: [batch_size, N*K]
        - outputs.shape: [batch_size, struct_size]
        '''
        logits = self.encode(inputs)
        latent_codes = self.gumbel_softmax(logits, temperature)
        outputs = self.decoder(latent_codes)
        return logits, latent_codes, outputs

# Train

In [5]:
batch_size = 100
epochs = 5_000
learning_rate = 0.001

T_init = 1.0                                  # initial temperature
T_min = 0.5                                   # minimum temperature
# T_rate = -np.log(T_min / T_init) / (epochs * (1/2))     # 指數模型
# T_rate = (T_min - T_init) / (epochs * (1/2))            # 線性模型
T_rate = 0.001
print(f"T_rate: {T_rate}")

K = 2     # number of classes
N = 64    # number of categorical distributions

T_rate: 0.001


In [6]:
# DataLoader
dataloader_train = DataLoader(dataset=dataset_train,
                              batch_size=batch_size,
                              shuffle=False)

In [7]:
# Loss Function
def cat_kl_div(logits):
    '''
    input
    - logits.shape: [batch_size, N*K]
    output
    - kl_loss.shape: []
    '''
    # Reshape to [batch_size*N, K]
    batch_size = len(logits)
    logits = logits.view(batch_size*N, K)
    
    q = dist.Categorical(logits=logits)
    p = dist.Categorical(probs=torch.full((batch_size*N, K), 1.0/K, device=device))    # uniform bunch of K-class categorical distributions
    kl = dist.kl.kl_divergence(q, p)    # kl.shape: [batch_size*N]
    kl_loss = torch.mean(kl)            # kl_loss.shape: []
    return kl_loss

def loss_func(outputs, labels, logits):
    '''
    input
    - outputs.shape: [batch_size, vec_size]
    - labels.shape: [batch_size, vec_size]
    - logits.shape: [batch_size, N*K]
    output
    - kl_loss.shape: []
    - rec_loss.shape: []
    - loss.shape: []
    '''
    
    rec_loss = nn.functional.binary_cross_entropy(outputs, labels, reduction="mean")
    
    kl_loss = cat_kl_div(logits)
    
    beta = 0.03
    loss = rec_loss + beta*kl_loss
    
    return rec_loss, beta*kl_loss, loss

In [8]:
model = CategoricalVAE(512, [300], N, K).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
temperature = T_init

In [ ]:
# Train
# 檢查梯度的啟用狀態
# for name, param in model.named_parameters():
#     print(f"Layer {name}: requires_grad = {param.requires_grad}")
# print("------------------------------------------")

model.train()

avg_rec_loss = []
avg_kl_loss  = []
avg_loss     = []

for epoch in range(epochs):
    
    total_rec_loss = 0
    total_kl_loss  = 0
    total_loss     = 0
    
    for data in dataloader_train:
        '''
        data.shape: [batch_size, num_of_channel=1, img_width=28, img_height=28]
        '''
        
        # Clear Gradients
        model.zero_grad()

        # Reshape the Input Images
        inputs = data
        '''
        inputs.shape: [batch_size, flatten_img_size]
        '''
        
        # Forward
        logits, latent_codes, outputs = model(inputs, temperature)
        '''
        - logits.shape: [batch_size, N*K]
        - latent_codes.shape: [batch_size, N*K]
        - outputs.shape: [batch_size, flatten_img_size]
        '''
        
        # 計算loss
        rec_loss, kl_loss, loss = loss_func(outputs, inputs, logits)    # 一個batch平均的loss
        total_rec_loss += rec_loss.item()
        total_kl_loss  += kl_loss.item()
        total_loss     += loss.item()
        
        # Backward & Update the parameters
        loss.backward()
        optimizer.step()

    # 看梯度
    # for name, param in model.named_parameters():
    #     if param.grad is not None:
    #         print(f"Layer {name}: Gradient norm: {param.grad.norm()}")
    #     else:
    #         print(f"Layer {name}: No gradient computed!")

    # T_next = T_init * np.exp(-T_rate*(epoch+1))    # 指數模型
    T_next = T_rate * epoch + T_init               # 線性模型
    temperature = np.maximum(T_next, T_min)
    # print(f"{epoch+1:>3}. temperature: {temperature:.2f}")
    
    # 一個epoch平均的loss
    avg_rec_loss.append( total_rec_loss / len(dataloader_train) )
    avg_kl_loss.append( total_kl_loss / len(dataloader_train) )
    avg_loss.append( total_loss / len(dataloader_train) )
    print(f"{epoch+1:>3}. avg_loss: {avg_loss[epoch]:.7f}, avg_rec_loss: {avg_rec_loss[epoch]:.7f}, avg_kl_loss: {avg_kl_loss[epoch]:.7f}")

  1. avg_loss: 0.6751169, avg_rec_loss: 0.6716433, avg_kl_loss: 0.0034735
  2. avg_loss: 0.5573539, avg_rec_loss: 0.5455156, avg_kl_loss: 0.0118384
  3. avg_loss: 0.4836924, avg_rec_loss: 0.4701264, avg_kl_loss: 0.0135661
  4. avg_loss: 0.4464288, avg_rec_loss: 0.4322750, avg_kl_loss: 0.0141539
  5. avg_loss: 0.4158098, avg_rec_loss: 0.4012841, avg_kl_loss: 0.0145256
  6. avg_loss: 0.3947001, avg_rec_loss: 0.3798896, avg_kl_loss: 0.0148104
  7. avg_loss: 0.3750573, avg_rec_loss: 0.3600522, avg_kl_loss: 0.0150051
  8. avg_loss: 0.3600102, avg_rec_loss: 0.3448574, avg_kl_loss: 0.0151528
  9. avg_loss: 0.3480897, avg_rec_loss: 0.3328007, avg_kl_loss: 0.0152890
 10. avg_loss: 0.3374164, avg_rec_loss: 0.3220178, avg_kl_loss: 0.0153986
 11. avg_loss: 0.3269475, avg_rec_loss: 0.3114642, avg_kl_loss: 0.0154833
 12. avg_loss: 0.3193054, avg_rec_loss: 0.3037537, avg_kl_loss: 0.0155517
 13. avg_loss: 0.3139624, avg_rec_loss: 0.2983434, avg_kl_loss: 0.0156190
 14. avg_loss: 0.3076694, avg_rec_loss

In [ ]:
x = np.linspace(0, epochs, epochs)

# 建立圖形和第一個y軸
fig, ax1 = plt.subplots(figsize=(7,5))
# 繪製第一組數據: avg_loss
ax1.plot(x, avg_loss, 'r-', label="loss")
ax1.set_xlabel('epoch')
ax1.set_ylabel('loss', color='r')
ax1.tick_params(axis='y', labelcolor='r')
# 強制 y1 使用科學記號
ax1.yaxis.set_major_formatter(ticker.ScalarFormatter(useMathText=True))
ax1.ticklabel_format(style='sci', axis='y', scilimits=(0, 0))  # 強制啟用科學記號
# ax1.get_yaxis().get_offset_text().set_position((, 0))  # 手動移動科學記號標示位置

# 建立第二個y軸: rec_loss
ax2 = ax1.twinx()
ax2.plot(x, avg_rec_loss, 'b--', label="rec_loss")
ax2.set_ylabel('rec_loss', color='b')
ax2.tick_params(axis='y', labelcolor='b')
# 強制 y2 使用科學記號
ax2.yaxis.set_major_formatter(ticker.ScalarFormatter(useMathText=True))
ax2.ticklabel_format(style='sci', axis='y', scilimits=(0, 0))  # 強制啟用科學記號
ax2.get_yaxis().get_offset_text().set_position((1.05, 0))  # 手動移動科學記號標示位置

# 建立第三個y軸: kl_loss
ax3 = ax1.twinx()
ax3.spines['right'].set_position(('outward', 70))  # 將第三個y軸移開一點
ax3.plot(x, avg_kl_loss, 'g-.', label="kl_loss")
ax3.set_ylabel('kl_loss', color='g')
ax3.tick_params(axis='y', labelcolor='g')
# 強制 y3 使用科學記號
ax3.yaxis.set_major_formatter(ticker.ScalarFormatter(useMathText=True))
ax3.ticklabel_format(style='sci', axis='y', scilimits=(0, 0))  # 強制啟用科學記號
ax3.get_yaxis().get_offset_text().set_position((1.28, 0))  # 手動移動科學記號標示位置

plt.xlim(left=0, right=epochs)
plt.ylim(bottom=0)

# 顯示圖例
fig.tight_layout()  # 確保佈局不重疊
plt.savefig("test.png")
plt.show()

# Test

In [ ]:
batch_size = 2000

In [ ]:
# DataLoader
dataloader_test = DataLoader(dataset=dataset_test,
                             batch_size=batch_size,
                             shuffle=True)

In [ ]:
# Show Image
plt.rcParams['figure.figsize'] = (10.0, 8.0)
plt.rcParams['image.interpolation'] = 'nearest'
plt.rcParams['image.cmap'] = 'gray'

def show_logits(logits):
    '''
    input
    - logits.shape: [batch_size, N*K]
    '''
    sqrtn = int(np.ceil(np.sqrt(logits.shape[0])))
    for index, logit in enumerate(logits):
        plt.subplot(sqrtn, sqrtn, index+1)
        plt.imshow(logit.reshape(N, K))
        plt.axis('off')
    plt.show()

In [ ]:
model.eval()

with torch.no_grad():
    
    for data in dataloader_test:
    
        inputs = data
        print("inputs.shape:", inputs.shape)
    
        # Inputs
        # for i, vector in enumerate(inputs):
        #     print(f"Batch {i+1}:")
        #     for j in range(0, len(vector), 10):
        #         print(vector[j:j+10].int().tolist())
        #     print("-" * 30)  # 分隔每個 batch
    
        # Inference/Forward
        logits, latent_code, outputs = model(inputs, temperature=0.01)
        '''
        - logits.shape: [batch_size, N*K]
        - latent_codes.shape: [batch_size, N*K]
        - outputs.shape: [batch_size, flatten_img_size]
        '''
        
        # Logits
        # print("logits.shape:", logits.shape)
        # show_logits(logits)

        # Latent Code
        # print("latent_code.shape:", latent_code.shape)
        # show_logits(latent_code)
        
        # Outputs
        outputs = (outputs>=0.5).int().float()
        # for i, vector in enumerate(outputs):
        #     print(f"Batch {i+1}:")
        #     for j in range(0, len(vector), 10):
        #         print(vector[j:j+10].int().tolist())
        #     print("-" * 30)  # 分隔每個 batch

        # 正確率
        equal_elements = inputs == outputs                          # 比對每個元素是否相等，結果是布林值的 tensor
        batch_equal_counts = equal_elements.sum(dim=1)              # 對每個 batch 計算相等的元素數量
        
        # print("每個 batch 中相等的比例")
        ratio = batch_equal_counts/512*100
        ratio = ratio.to("cpu").numpy()
        # for i in range(len(ratio)):
        #     print(f"{ratio[i]:.0f}", end=', ')
        #     if i % 10 == 9:
        #         print("")
        avg_accuracy = np.mean(ratio)    # 對batch取平均
        std_accuracy = np.std(ratio)
        print(f"正確率的平均值：{avg_accuracy:.2f}%")
        print(f"正確率的標準差：{std_accuracy:.2f}%")

In [ ]:
# 畫出等效圖
k = 2
equal_elements = inputs[k] == outputs[k]
equal_elements = equal_elements.sum()
print(equal_elements)
print(f"ratio: {equal_elements/512*100}%")
rings_utils.plot_rings(inputs[k], save=False)
rings_utils.plot_rings(outputs[k], save=False)


# Generation

In [ ]:
# 第一個criteria：latent vector變動一點點，output應該也要只能變動一點點

def hamming_dist(str1, str2):
    if len(str1) != len(str2):
        raise ValueError("兩個字串的長度必須相同")
    return sum(c1 != c2 for c1, c2 in zip(str1, str2))

def eval_1(latent_code):
    '''
    latent_code變動一點，看輸出變動多少
    K=2，且要已經binarized
    latent_code.shape: [batch_size, vec_size]    
    '''
    batch_size = len(latent_code)
    vec_size   = len(latent_code[0])

    # 變動一個latent_code的元素（要把它拆成兩半，另一半也要變）
    latent_code_new = latent_code.clone()
    for i in range(batch_size):
        idx_front = random.randint(1, vec_size//2) - 1
        idx_rear  = idx_front + vec_size//2
        latent_code_new[i][idx_front] = 0.0 if latent_code_new[i][idx_front] == 1.0 else 1.0
        latent_code_new[i][idx_rear]  = 0.0 if latent_code_new[i][idx_rear] == 1.0 else 1.0

    # 計算輸出的差異
    outputs     = model.decode(latent_code)
    outputs     = (outputs>=0.5).int().float()
    outputs_new = model.decode(latent_code_new)
    outputs_new = (outputs_new>=0.5).int().float()
    
    diff = []
    for i in range(batch_size):
        diff.append(hamming_dist(outputs[i], outputs_new[i]) / 512 * 100)    # 不同輸出的比例%
    diff = np.array( [tensor.item() for tensor in diff] )    # diff.shape: [batch_size]

    # 畫環
    k=1
    rings_utils.plot_rings(outputs[k], save=False)
    rings_utils.plot_rings(outputs_new[k], save=False)
    
    avg_diff = np.mean(diff)
    std_diff = np.std(diff)
    return avg_diff, std_diff
    
    # # 畫直方圖統計
    # bins = range(0, 100, 1)  # 從0到30，每5為一個區間
    
    # # 繪製直方圖
    # plt.hist(diff, bins=bins, edgecolor='black')
    # plt.title('Histogram of diff')
    # plt.xlabel('Value Range')
    # plt.ylabel('Frequency')
    # plt.xticks(bins)  # 設置x軸的刻度
    # plt.grid(axis='y', linestyle='--', alpha=0.7)
    # plt.show()

In [ ]:
# 第二個criteria：假設rec_loss夠好，隨機抽樣一筆latent vector再經過decoder, encoder回來得到的latent vector比對後就可以評斷decoder的好壞

def eval_2(latent_code):
    pass
    # inputs = outputs
    # latent_code_new = model.encode(inputs)

    # latent_code_new = (latent_code_new>=0.5).int().float()
    # # 正確率
    # equal_elements = latent_code == latent_code_new             # 比對每個元素是否相等，結果是布林值的 tensor
    # batch_equal_counts = equal_elements.sum(dim=1)              # 對每個 batch 計算相等的元素數量
    # print("每個 batch 中相等的元素數量：", batch_equal_counts)
    # print("每個 batch 中相等的元素比例：", batch_equal_counts/len(latent_code[0]))

In [ ]:
# 第三個criteria：判斷書出的環數有沒有小於threshold（但是有可能小小的出錯就讓他多了很多等校環）
def eval_3(latent_code):
    '''
    input
    - latent_code.shape: [batch_size, vec_size]
    '''
    outputs = model.decode(latent_code)
    outputs = (outputs>=0.5).int().float()
    
    nums = []    # 紀錄每個batch的等效環數
    for i in range(len(latent_code)):
        nums.append( rings_utils.equiv_rings(outputs[i]) )
        # print(f"等效環數：{nums[i]}")
    nums = np.array(nums)
    
    avg_nums = np.mean(nums)    # 等效環數的平均值
    std_nums = np.std(nums)     # 等效環數的標準差
    return avg_nums, std_nums

In [ ]:
batch_size = 1000
temperature = 0.01
with torch.no_grad():
    logits = torch.rand((batch_size, N, K), device=device)
    latent_code = model.gumbel_softmax(logits, temperature)
    latent_code = (latent_code>=0.5).int().float()

    # 評估生成模型的好壞
    avg_diff, std_diff = eval_1(latent_code)
    print(f"變化比例的平均值：{avg_diff:.2f}%")    # 正常情況應該10%左右
    print(f"變化比例的標準差：{std_diff:.2f}%")    # 應該要盡量小ㄅ

    print("------------------------------------------")
    
    avg_nums, std_nums = eval_3(latent_code)
    print(f"等效環數的平均值：{avg_nums:.2f}")    # 正常情況應該要小於50，極限就是超過50一點
    print(f"等效環數的標準差：{std_nums:.2f}")    # 應該要盡量小ㄅ